# CN7 불량 시기 효과크기 (보고서 제3장 <표 6>, 제4장 <표 7> CN7 행)

- 입력: `../1_CN7/data/moldset_labeled_cn7.csv` (labeled 원자료, 역변환 전 z값)
- 샷 정의는 `1_CN7/model_zoo.load_evalset`과 같다: 공정 값 조합이 같은 두 행 = 한 샷, 처음 나온 순서가 샷 번호
- 효과크기 = (두 집단 샷 평균 차이) ÷ (전체 샷 표준편차). 샷 단위로 계산해 같은 샷의 두 부품이 두 번 세어지지 않게 한다

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import beta

TARGET = 'PassOrFail'
lab = pd.read_csv('../1_CN7/data/moldset_labeled_cn7.csv', index_col=0)
feats = [c for c in lab.columns if c != TARGET and lab[c].nunique() > 1]
grp = lab.groupby(feats, sort=False)
ev = lab[feats + [TARGET]].assign(shot_id=grp.ngroup())
shots = ev.groupby('shot_id').agg({**{f: 'first' for f in feats}, TARGET: 'max'})
print('샷', len(shots), '/ 불량 샷', shots[TARGET].sum(), '/ 불량 샷 번호', shots.index[shots[TARGET] == 1].tolist())

## 불량 시기(샷 0~60) vs 이후 구간

In [ ]:
last_ng = shots.index[shots[TARGET] == 1].max()
period = shots.index <= last_ng
es = ((shots.loc[period, feats].mean() - shots.loc[~period, feats].mean()) / shots[feats].std())
table6 = es.reindex(es.abs().sort_values(ascending=False).index).head(8).round(2).rename('효과크기 (SD)')
table6

## 불량 시기 안: 불량 샷 vs 정상 샷 (불량 샷 14개라 참고용)

In [ ]:
A = shots[period]
es2 = (A.loc[A[TARGET] == 1, feats].mean() - A.loc[A[TARGET] == 0, feats].mean()) / A[feats].std()
es2.reindex(es2.abs().sort_values(ascending=False).index).head(6).round(2)

## 참고: 라벨 구간 처음 N샷 전수 검사 (제4장 <표 7> 참고 행)

표본 안에서 관측한 값이며, 구간(샷 0~60)은 불량 위치를 본 뒤 정한 것이다. 신뢰구간은 샷 단위 Clopper–Pearson 95%.
**운영 규칙으로 쓸 수 없다**: 비라벨 데이터 기준 라벨 샷 0(비라벨 인덱스 751,091)은 직전 큰 중단·조건 변경(인덱스 692,358→735,217)보다 약 440샷 뒤의 생산 중간이다.

In [ ]:
rows = []
for n in [30, 61, 100]:
    insp = ev['shot_id'] < n
    caught_parts = ev.loc[insp, TARGET].sum()
    caught_shots = int(shots.loc[shots.index < n, TARGET].sum())
    k, m = caught_shots, int(shots[TARGET].sum())
    lo = beta.ppf(0.025, k, m - k + 1) if k > 0 else 0.0
    rows.append({'처음 N샷': n, '검사율': round(insp.mean(), 3), '포착 불량 부품': f'{caught_parts}/{ev[TARGET].sum()}',
                 '포착 불량 샷': f'{k}/{m}', '재현율 95% 하한 (샷 기준)': round(lo, 2)})
pd.DataFrame(rows)